In [5]:
import duckdb
import random
from functions.analyte import ANALYTES
from networks.cnn_deep import CNNModel
from networks.auto_encoder import AutoencoderModel
from functions.evaluation import evaluate
from functions.full_model import predict


con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, age, gender, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set
                 FROM protein_data 
                 WHERE set = 'test'
                 AND observation_nr = 1
                 """).df()

df = predict(df)
con.close()


In [6]:
import numpy as np

df['type'] = 'normal'

#slår ihop oligoklonalt och lätt avvikande
df.loc[df['label']==4,'label'] = 2
df.loc[df['final_prediction']==4,'final_prediction'] = 2


mat = np.zeros((3,3))
total_missclassified = sum(df['label'] != df['final_prediction'])
for i in range(3):
    for j in range(3):
        mat[i,j] = sum((df['label'] == i) & (df['final_prediction'] == j))
        #ids |= set(df.loc[(df['label'] == i) & (df['final_prediction'] == j),'row_id' ].sample(int(np.ceil(mat[i,j]))))

ids = set(df.loc[(df['label'] == 1) & (df['final_prediction'] == 0),'row_id' ]) 
ids |= set(df.loc[(df['label'] == 1) & (df['final_prediction'] == 2),'row_id' ])


ids |= set(df.loc[(df['label'] == 0) & (df['final_prediction'] == 1),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 0) & (df['final_prediction'] == 2),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 2) & (df['final_prediction'] == 0),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] == 2) & (df['final_prediction'] == 1),'row_id' ].sample(21))
ids |= set(df.loc[(df['label'] != df['final_prediction']),'row_id' ].sample(1))
df.loc[df['row_id'].isin(ids),'type'] = 'difficult'


ids |= set(df.loc[(df['label'] == 0) & (~df['row_id'].isin(ids)), 'row_id'].sample(50,random_state=42)) 
ids |= set(df.loc[(df['label'] == 1) & (~df['row_id'].isin(ids)), 'row_id'].sample(50,random_state=43))


print(df[df['type'] == 'difficult'].shape[0])

print(len(ids))
print(ids)

100
200
{42499, 77317, 143369, 132107, 100876, 26129, 29202, 95762, 128532, 148501, 92182, 113687, 93720, 133656, 144915, 53270, 181279, 156196, 91178, 140335, 8240, 122416, 150066, 115253, 2614, 96826, 20026, 103495, 3146, 57419, 29260, 109131, 162390, 148056, 106072, 96344, 101467, 47200, 2659, 33898, 58988, 125550, 77426, 128115, 115, 49781, 84086, 106615, 110200, 133241, 111737, 154231, 89723, 54909, 94843, 67705, 30336, 153217, 171650, 52354, 35462, 160904, 172680, 90766, 117908, 43158, 71320, 28313, 180383, 124579, 92326, 90792, 176809, 150186, 117423, 58035, 42169, 15033, 110778, 62143, 128703, 177345, 71875, 116420, 11973, 9923, 122572, 160463, 6351, 70354, 90322, 160980, 46809, 163035, 47323, 168157, 52957, 125666, 118499, 32486, 24811, 185079, 140025, 176899, 96516, 105735, 39178, 119052, 105229, 110348, 20753, 58129, 84756, 155926, 160536, 86809, 159007, 118052, 127787, 110380, 25899, 106798, 83256, 3386, 41277, 99138, 122694, 127302, 112969, 34635, 26958, 159055, 157520, 84

In [8]:
df = df[df['row_id'].isin(ids)]
df = df.sample(frac=1).reset_index(drop=True)
print(len(df))


df = df[['row_id','type','value','albumin','antitrypsin','orosomukoid','haptoglobin','crp','igg','iga','igm']]
con = duckdb.connect('../application/application.db')
con.execute('DROP TABLE IF EXISTS difficult_cases')
con.execute('CREATE TABLE difficult_cases AS SELECT * FROM df')

con.close()


200


IOException: IO Error: Could not set lock on file "/Users/assarleideman/exjobb/application/application.db": Conflicting lock is held in /opt/homebrew/Cellar/duckdb/1.5.0/bin/duckdb (PID 81666) by user assarleideman. See also https://duckdb.org/docs/stable/connect/concurrency